# Basic prompt engineering

## Imports

In [21]:
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

from pathlib import Path
from typing import Any
import re
import json


## Configs

In [35]:
# Model and adapter paths
MODEL_PATH = Path("../data/models/gemma-2b/")
LORA_ADAPTER_PATH = Path("../data/gemma-2b-alpaca-lora-final/")
QWEN_CODER_WITHOUT_ADAPTER = Path("/home/nguyen/.cache/huggingface/hub/models--unsloth--Qwen2.5-Coder-1.5B-bnb-4bit/snapshots/8e7c25d88b601ed8c67058751f5f6bd03f7538a8")

In [36]:
model_path = QWEN_CODER_WITHOUT_ADAPTER

tokenizer = AutoTokenizer.from_pretrained(model_path)

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    device_map="auto"
)

Loading weights: 100%|██████████| 338/338 [00:01<00:00, 323.88it/s]


## Utilities

### Load_local_model

In [ ]:
def load_local_model(
    model_path: Path,
    adapter_path: Path | None = None,
    custom_path_without_adapter = None,
) -> tuple[Any, Any]:
    """Load a local causal language model and tokenizer."""

    if not model_path.exists():
        raise FileNotFoundError(
            f"Model path not found: {model_path.resolve()}\n"
            "Download the base model first or update MODEL_PATH."
        )

    # 4-bit quantization reduces GPU memory usage so a smaller GPU can run the model.
    # NF4 is commonly used for LLM inference/fine-tuning because it preserves quality well.
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
    )

    # device_map="auto" lets Transformers place modules on available hardware.
    # If you only want CUDA, use device_map="cuda" like your previous notebook cell.
    model = AutoModelForCausalLM.from_pretrained(
        model_path,
        quantization_config=quantization_config,
        device_map="auto",
    )

    if adapter_path is not None and adapter_path.exists():
        # A LoRA adapter stores small task-specific weight updates on top of the base model.
        model = PeftModel.from_pretrained(model, adapter_path)
    elif adapter_path is not None:
        print(f"LoRA adapter path not found, using base model only: {adapter_path.resolve()}")

    tokenizer = AutoTokenizer.from_pretrained(model_path)

    # Some decoder-only models do not define a pad token by default.
    # Reusing eos_token avoids generation errors when padding is needed.
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model.eval()
    return model, tokenizer


model, tokenizer = load_local_model(MODEL_PATH, LORA_ADAPTER_PATH)
print("Model and tokenizer loaded successfully.")

Loading weights:   1%|          | 1/164 [00:00<01:57,  1.39it/s]/home/nguyen/micromamba/envs/llm_env/lib/python3.11/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100%|██████████| 164/164 [00:03<00:00, 45.09it/s]
/home/nguyen/micromamba/envs/llm_env/lib/python3.11/site-packages/peft/peft_model.py:622: UserWarning: Found missing adapter keys while loading the checkpoint: ['base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight', 'base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight', 'base_model.model.model.layers.1.self_attn.q_proj.lora_A.default.weight', 'base_model.model.model.layers.1.self_attn.q_proj.lora_B.def

Model and tokenizer loaded successfully.


In [68]:
# =========================================================
# 2. DUMMY WEB SEARCH TOOL
# =========================================================

def web_search(query: str):
    """
    Dummy search function.
    """

    print(f"\n[TOOL CALLED] web_search('{query}')\n")

    return {
        "query": query,
        "results": [
            "The weather today is sunny and warm.",
            "Vietnamese weather forcast says today Vietnam is sunny",
            "In Vietname, it's rainy this morning and now at afternoon, it's cloudy",
            "The weather tomorrow in Vietnam is cloudy."
        ]
    }

In [32]:
# =========================================================
# 3. TOOL SCHEMA GIVEN TO MODEL
# =========================================================

TOOLS = """
You are an AI assistant with access to tools.

If the user asks for:
- current information
- weather
- news
- internet knowledge

you MUST call web_search.

Reply ONLY in JSON.

Example:

{
  "tool": "web_search",
  "query": "weather today in Vietnam"
}
"""

In [51]:
# =========================================================
# 4. GENERATION FUNCTION
# =========================================================

def generate(prompt, max_new_tokens=200):

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            do_sample=False, # do_sample=False = always pick the most likely next token; do_sample=True = randomly sample possible next tokens.

            # EOS
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    text = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return text

In [25]:

# =========================================================
# 5. TOOL CALL DETECTOR
# =========================================================

def try_extract_tool_call(text):

    match = re.search(r'\{.*\}', text, re.DOTALL)

    if not match:
        return None

    try:
        data = json.loads(match.group())

        if data.get("tool") == "web_search":
            return data

    except:
        pass

    return None

print(
    try_extract_tool_call(
        '{ "tool": "web_search", "query": "What is Gemma?" }'
    )
)

{'tool': 'web_search', 'query': 'What is Gemma?'}


In [63]:
# =========================================================
# 6. SIMPLE AGENT LOOP
# =========================================================

user_input = "Search for the newest weather forcast for today in Vietnam"

messages = f"""
{TOOLS}

User:
{user_input}

Assistant:
"""

# FIRST MODEL CALL
response = generate(messages)

print("\n========== MODEL RESPONSE ==========\n")
print(response)


[transformers] Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



========== MODEL RESPONSE ==========

{
  "tool": "web_search",
  "query": "newest weather forcast for today in Vietnam"
}


In [64]:
# TRY DETECT TOOL CALL
tool_call = try_extract_tool_call(response)
tool_call

{'tool': 'web_search', 'query': 'newest weather forcast for today in Vietnam'}

In [69]:
if tool_call:

    # EXECUTE TOOL
    tool_result = web_search(tool_call["query"])

    # FEED TOOL RESULT BACK TO MODEL
    second_prompt = f"""
You are an AI assistant.

TOOLS:
{TOOLS}

User:
{user_input}

Assistant called tool:
{json.dumps(tool_call, indent=2)}

Tool returned:
{json.dumps(tool_result, indent=2)}

Using the tool result, answer the user directly.

Assistant:
"""
    final_response = generate(second_prompt)

    print("\n========== FINAL ANSWER ==========\n")
    print(final_response)

[transformers] Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



[TOOL CALLED] web_search('newest weather forcast for today in Vietnam')



/home/nguyen/micromamba/envs/llm_env/lib/python3.11/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)



========== FINAL ANSWER ==========

{
  "answer": "The weather today is sunny and warm. Vietnamese weather forcast says today Vietnam is sunny. In Vietname, it's rainy this morning and now at afternoon, it's cloudy. The weather tomorrow in Vietnam is cloudy."
}
